# 分子内接触

读取计算结果绘图，不读取轨迹或重新计算接触。接触规则以结果 metadata 中实际参数为准。
接触图对每帧的链平均等权；浓相为空的帧排除，并报告有效帧数。残基位置为链内零起始索引。
修改距离或序列排除规则后，请先重新运行分子内计算。


In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Find the project among the working directory and its parents.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "simulations.yaml").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the project, or set ROOT explicitly")
sys.path.insert(0, str(ROOT))
import project
import result_files as files

CONFIG = ROOT / "simulations.yaml"
SELECTED = None  # None means all registered simulations.
OUTPUT = ROOT / "02_calculate/intramolecular/output"
SIMULATIONS = project.select_simulations(CONFIG, SELECTED)
RESULTS = {s["id"]: files.completed(OUTPUT / s["id"]) for s in SIMULATIONS}
SOURCES = {key: files.read_json(path / "metadata.json") for key, path in RESULTS.items()}
COLORS = {name: plt.get_cmap("tab10")(i % 10) for i, name in enumerate(RESULTS)}
FIGURES = OUTPUT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "svg.fonttype": "none", "pdf.fonttype": 42})

def export(fig, name):
    for key, path in RESULTS.items():
        files.completed(path)
        if files.read_json(path / "metadata.json") != SOURCES[key]:
            raise ValueError("Results changed; restart and rerun the notebook")
    for extension in ("png", "pdf", "svg"):
        fig.savefig(FIGURES / f"{name}.{extension}", dpi=300, bbox_inches="tight")
    files.write_json(FIGURES / f"{name}.sources.json", {
        key: files.read_json(path / "metadata.json") for key, path in RESULTS.items()})


In [ ]:
frames = pd.concat([pd.read_csv(p / "frame_means.csv") for p in RESULTS.values()], ignore_index=True)
summary = pd.concat([pd.read_csv(p / "summary.csv") for p in RESULTS.values()], ignore_index=True)
display(summary)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), constrained_layout=True)
for scope, ax in zip(("all", "dense"), axes):
    for name, group in frames[frames.scope.eq(scope)].groupby("simulation_id", sort=False):
        ax.plot(group.frame, group.contact_count, color=COLORS[name], label=name, alpha=.8)
    ax.set(xlabel="Analysis frame (0-based)", ylabel="Intrachain contact pairs per chain", title=scope)
axes[0].legend()
export(fig, "intrachain_contact_counts")
plt.show()


In [ ]:
fig, axes = plt.subplots(len(RESULTS), 2, figsize=(8, 3.5 * len(RESULTS)),
                         squeeze=False, constrained_layout=True)
for row, (name, path) in enumerate(RESULTS.items()):
    with np.load(path / "contact_map.npz") as saved:
        for col, scope in enumerate(("all", "dense")):
            image = axes[row, col].imshow(saved[f"{scope}_probability"], vmin=0, vmax=1,
                                          cmap="viridis", origin="lower")
            valid = int(saved[f"{scope}_valid_frames"])
            axes[row, col].set(title=f"{name} · {scope} · {valid} valid frames",
                               xlabel="Residue index", ylabel="Residue index")
fig.colorbar(image, ax=axes.ravel().tolist(), label="Contact probability (frame-equal)")
export(fig, "intrachain_contact_maps")
plt.show()


## 瞬时接触网络指标
每条链分别计算，再进行帧内链平均、时间平均。主链不作为接触边。
最大接触簇占比不是折叠比例；全局效率仅描述接触路径。接触数量也会影响它们。


In [ ]:
metrics = {"contact_count": "Contact pairs / chain",
           "largest_component_fraction": "Largest contact cluster / residues",
           "global_efficiency": "Contact-network global efficiency"}
fig, axes = plt.subplots(2, 3, figsize=(13, 6), constrained_layout=True)
for row, scope in enumerate(("all", "dense")):
    for ax, (metric, label) in zip(axes[row], metrics.items()):
        for name, group in frames[frames.scope.eq(scope)].groupby("simulation_id", sort=False):
            ax.plot(group.frame, group[metric], color=COLORS[name], label=name, alpha=.8)
        ax.set(xlabel="Analysis frame", ylabel=label, title=scope)
axes[0, 0].legend()
export(fig, "intrachain_network_metrics")
plt.show()


In [ ]:
degree = pd.concat([pd.read_csv(p / "degree_distribution/summary.csv")
                    for p in RESULTS.values()], ignore_index=True)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), constrained_layout=True)
for scope, ax in zip(("all", "dense"), axes):
    for name, group in degree[degree.scope.eq(scope)].groupby("simulation_id", sort=False):
        ax.plot(group.degree, group.fraction_mean_frames, color=COLORS[name], label=name)
    ax.set(xlabel="Residue contact degree k", ylabel="P(k), frame-equal", title=scope)
axes[0].legend()
export(fig, "intrachain_degree_distribution")
plt.show()


## 序列弧线快照
明确选择一帧、一条链；默认第一帧第 0 条链，并不保证统计代表性。
节点按序列排列，弧线跨度表示序列间隔，不表示空间距离。灰色主链仅供定位。
无接触时仍显示全部节点。浓相归属读取计算产物；不会重新计算接触。


In [ ]:
from matplotlib.patches import Arc

SNAPSHOT_SIMULATION = next(iter(RESULTS))
SNAPSHOT_FRAME = 0
SNAPSHOT_CHAIN = 0
path = RESULTS[SNAPSHOT_SIMULATION]
for chunk_path in sorted((path / "edges").glob("*.npz")):
    with np.load(chunk_path) as saved:
        matches = np.flatnonzero(saved["frame"] == SNAPSHOT_FRAME)
        if not len(matches):
            continue
        n_residues = int(saved["n_residues"])
        dense_mask = saved["dense_chains"][matches[0]]
        if not 0 <= SNAPSHOT_CHAIN < len(dense_mask):
            raise ValueError("Snapshot chain out of range")
        edges = saved["edges"]
        edges = edges[(edges[:, 0] == SNAPSHOT_FRAME) & (edges[:, 1] == SNAPSHOT_CHAIN)]
        is_dense = bool(dense_mask[SNAPSHOT_CHAIN])
        break
else:
    raise ValueError("Snapshot frame not found")
fig, ax = plt.subplots(figsize=(12, 3), constrained_layout=True)
x = np.arange(n_residues)
ax.plot(x, np.zeros(n_residues), color=".8", linewidth=1, zorder=0)
ax.scatter(x, np.zeros(n_residues), s=7, color=".3")
max_span = 0
for _, _, i, j in edges:
    left, right = sorted((int(i), int(j)))
    span = right - left
    max_span = max(max_span, span)
    ax.add_patch(Arc(((left + right) / 2, 0), span, span * .5,
                     theta1=0, theta2=180, color=COLORS[SNAPSHOT_SIMULATION], alpha=.35, lw=.7))
ax.set(xlim=(-1, n_residues), ylim=(-1, max(2, max_span * .27)),
       xlabel="Residue index (0-based)", yticks=[],
       title=f"{SNAPSHOT_SIMULATION}: frame {SNAPSHOT_FRAME}, chain {SNAPSHOT_CHAIN}, dense={is_dense}")
export(fig, f"intrachain_arcs_{SNAPSHOT_SIMULATION}_{SNAPSHOT_FRAME}_{SNAPSHOT_CHAIN}")
plt.show()
